# COCO Segmentation DataLoader Demo
This notebook demonstrates the newly implemented `COCOSegmentationDataset` and how it handles polygon annotations through the refactored inheritance structure.

In [ ]:
%reset -f
%load_ext autoreload
%autoreload 2

%reload_ext autoreload

In [ ]:
import torch
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print(f"Current working directory: {os.getcwd()}")
from omegaconf import OmegaConf
from yolo.config.config import DataConfig, DatasetConfig
from yolo.data.loader import create_dataloader
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import numpy as np
from pathlib import Path
from yolo.utils.drawer import draw_masks, draw_bboxes


## 1. Load Configurations
We use the same `coco.yaml` but initialize it for a segmentation task.

In [ ]:
# Load the dataset config (COCO)
coco_yaml_path = "/home/shrey/projects/yolo/yolo/config/dataset/coco.yaml"
dataset_cfg_dict = OmegaConf.to_container(OmegaConf.load(coco_yaml_path), resolve=True)
dataset_cfg = DatasetConfig(**dataset_cfg_dict)

# Set up the DataConfig
data_cfg = DataConfig(
    shuffle=True,
    batch_size=4,
    pin_memory=False,
    dataloader_workers=0,
    image_size=[640, 640],
    data_augment={},
    source=None,
    dynamic_shape=False
)

print(f"✅ Loading segmentation dataset from: {dataset_cfg.path}")

## 2. Initialize Segmentation DataLoader
Calling `create_dataloader` with `task="segmentation"` will now automatically use `COCOSegmentationDataset`.

In [ ]:
dataloader = create_dataloader(data_cfg, dataset_cfg, task="segment")
print("✅ Segmentation DataLoader created successfully.")

## 3. Fetch and Inspect a Batch
Notice the new `batch.masks` attribute which contains the polygon data.

In [ ]:
batch = next(iter(dataloader))

print(f"Batch object: {type(batch)}")
print(f"Images shape: {batch.images.shape}")
print(f"Targets (Boxes) shape: {batch.targets.shape}")
print(f"Masks type: {type(batch.masks)} (List of polygons per image)")

if batch.masks:
    num_masks = len(batch.masks[0]) if batch.masks[0] is not None else 0
    print(f"Number of masks in first image: {num_masks}")

## 4. Visualization of Polygons
We render the polygons directly onto the image to verify they are loaded and scaled correctly.

In [ ]:
num_to_show = 4
fig, axes = plt.subplots(1, num_to_show, figsize=(20, 6))

for i in range(num_to_show):
    print(batch.images[i].shape)
    img_with_masks = draw_masks(
        batch.images[i], 
        batch.masks[i], 
        idx2label=dataset_cfg.class_list
    )
    
    axes[i].imshow(img_with_masks)
    axes[i].set_title(f"File: {Path(batch.paths[i]).name}", fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.show()


## Standalone Transform Testing (with Masks)

We can also test transforms with segmentation masks. The `BaseTransform` ensures that when an image is flipped, cropped, or resized, the corresponding masks are synchronized perfectly.

In [ ]:
import torch
sample_idx = 5
image, _ = dataloader.dataset.get_image(sample_idx)
masks = dataloader.dataset.get_labels(sample_idx)

# Derive bboxes from polygons
bboxes = []
for poly in masks:
    cls = poly[0]
    pts = poly[1:].reshape(-1, 2)
    bboxes.append([cls, pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max()])
boxes = torch.tensor(bboxes).reshape(-1, 5)

In [ ]:
from yolo.data.augmentation.transforms import HorizontalFlip, RandomCrop, Mosaic, MixUp, VerticalFlip, PadAndResize

def plot_transform_with_masks(aug, title):
    # original image with masks and boxes
    orig_img = image.copy()
    orig_boxes = boxes.clone()
    
    # Draw original masks
    orig_res = orig_img
    if masks:
        # deepcopy masks to avoid drawing mutability issues if any
        orig_res = draw_masks(orig_res, [m.clone() for m in masks])
        
    # Draw original bboxes
    w_orig, h_orig = orig_res.size
    orig_pixel_boxes = orig_boxes.clone()
    orig_pixel_boxes[:, [1, 3]] *= w_orig
    orig_pixel_boxes[:, [2, 4]] *= h_orig
    orig_res = draw_bboxes(orig_res, orig_pixel_boxes, idx2label=dataset_cfg.class_list)

    # Perform Augmentation
    res = aug(image.copy(), boxes.clone(), [m.clone() for m in masks] if masks else None)
    aug_img = res[0]
    aug_boxes = res[1]
    aug_masks = res[2]
    
    # Draw aug masks
    res_img = aug_img
    if aug_masks:
        res_img = draw_masks(res_img, aug_masks)
        
    # Draw aug bboxes
    w, h = aug_img.size
    pixel_boxes = aug_boxes.clone()
    if pixel_boxes.numel() > 0:
        pixel_boxes[:, [1, 3]] *= w
        pixel_boxes[:, [2, 4]] *= h
    res_img = draw_bboxes(res_img, pixel_boxes, idx2label=dataset_cfg.class_list)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(orig_res)
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    axes[1].imshow(res_img)
    axes[1].set_title(title)
    axes[1].axis('off')
    plt.show()

In [ ]:
# Test Horizontal Flip with Masks
plot_transform_with_masks(HorizontalFlip(prob=1.0), "Horizontal Flip + Masks")

In [ ]:
# Test Vertical Flip with Masks
plot_transform_with_masks(VerticalFlip(prob=1.0), "Vertical Flip + Masks")

In [ ]:
# Test PadAndResize with Masks
plot_transform_with_masks(PadAndResize(image_size=(640, 640)), "Pad and Resize + Masks")

In [ ]:
# Test Random Crop with Masks
plot_transform_with_masks(RandomCrop(prob=1.0), "Random Crop + Masks")

In [ ]:
# Test Mosaic (4-image composition with masks)
mosaic = Mosaic(prob=1.0).set_parent(dataloader.dataset.transform)
plot_transform_with_masks(mosaic, "Mosaic (4-image Grid) + Masks")

In [ ]:
# Test MixUp with Masks
mixup = MixUp(prob=1.0).set_parent(dataloader.dataset.transform)
plot_transform_with_masks(mixup, "MixUp + Masks")